# `contextual-turn-encoder-base` **v3** — entrenamiento en M2

**BERT-base, literal, sobre turnos.** Misma arquitectura BERT-fiel que v2 pero a **tamaño BERT-base — 12 capas / 12 heads (head_dim 64) / intermediate 3072** — y entrenada con la **receta de pretraining de BERT**: `lr=1e-4`, AdamW **sin weight decay en bias/LayerNorm** (`eps=1e-6`), warmup + decay lineal del LR a 0. Es el **punto de partida fiel** desde el cual innovar (Fase 2: codebook).

> Tres modelos, ningún registro se pisa:
> - **v1** — arquitectura custom (pre-LN + residual), receta propia (`lr 2e-4`). *(notebook 02)*
> - **v2** — arquitectura **BERT-fiel** (post-LN), **receta del v1** (`lr 2e-4`, 6 capas/8 heads). *(notebook 03)*
> - **v3** — **BERT-base literal** (12 capas/12 heads), **receta de BERT** (`lr 1e-4`, no-decay en bias/LayerNorm). *(notebook 04)*
>
> En `config.json`, `arch` distingue solo **arquitectura** → v1=`"v1"`, **v2 y v3 son `"v2"`** (misma estructura
> de módulos); lo que los separa es **tamaño + receta**, anotado en `trainlog.jsonl` (`tag`, `recipe`).
> Divergencias vs BERT en [`docs/model/v2.md`](../../docs/model/v2.md).

## 1. Setup (rutas + identidad de la corrida)

In [ ]:
import os, sys, json, time
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

# rutas del repo en tu Mac (ya apuntan al lugar correcto; cambialas solo si moviste el repo)
REPO = os.path.expanduser("~/Documents/GitHub/doctorado-unsl")
PKG = os.path.join(REPO, "packages/contextual-turn-embeddings")
RECIPE = os.path.join(PKG, "training/contextual-turn-encoder-base")
CONFIG_PATH = os.path.join(RECIPE, "config.yaml")
ANN = os.path.expanduser("~/Documents/GitHub/ANN-UNSL")
assert os.path.isdir(REPO) and os.path.isdir(ANN), f"Revisá las rutas: REPO={REPO} | ANN={ANN}"

# ===== identidad de ESTA corrida =====
VER = "v3"                  # "v2" | "v3"  -> nombre de carpeta de checkpoints
LR = 1e-4                      # v2: 2e-4 (receta v1) | v3: 1e-4 (receta BERT)
EPOCHS = 15              # budget del schedule (LR decae a 0 al final)
NLAYERS_OVERRIDE = 12   # v2: 6 | v3: 12 (BERT-base)
NHEADS_OVERRIDE = 12     # v2: 8 | v3: 12 (head_dim 64, BERT-base)
BERT_RECIPE = True    # v3: True -> AdamW no-decay en bias/LayerNorm, eps=1e-6

CORPUS = "full"                  # "full" | "1m"  (la definitiva va sobre full)
if CORPUS == "full":
    META_PATH = os.path.join(REPO, "data/d2f-full/dialogs-full.pkl")
    BASE_EMB_PATH = os.path.join(REPO, "data/d2f-full/base_embeddings.npy")
else:                            # 1m = colección del benchmark (dialogs-2.0)
    META_PATH = os.path.join(ANN, "data/dialogs-2.0.pkl")
    BASE_EMB_PATH = os.path.join(ANN, "data/embeddings_dialog2flow.npy")
HELDOUT_META = os.path.join(ANN, "data/dialogs-2.0.pkl")   # held-out reproducible (semilla 42)
OUT = os.path.join(PKG, "models")                          # gitignored

sys.path.insert(0, RECIPE)
import heldout as H
from contextual_turn_embeddings import (Config, DialogueDataset, EmbeddingRetrievalConfig,
    build_model, collate_dialogues, compute_objectives, get_device, resolve_losses_for_mode, set_seed)
from contextual_turn_embeddings.train import build_linear_warmup_scheduler
print("device:", get_device("mps"), "| VER:", VER, "| LR:", LR, "| capas/heads:",
      NLAYERS_OVERRIDE, "/", NHEADS_OVERRIDE, "| BERT_RECIPE:", BERT_RECIPE)

## 2. Cargar datos (memmap) y excluir held-out

In [ ]:
df = pd.read_pickle(META_PATH)[["dialogue_id", "turn_id", "speaker", "utterance"]].copy()
df["row_id"] = np.arange(len(df), dtype=np.int64)          # = índice en el memmap

emb = np.load(BASE_EMB_PATH, mmap_mode="r")                # NO entra en RAM
assert len(df) == len(emb), (len(df), len(emb))

df_1m = pd.read_pickle(HELDOUT_META)
heldout_ids = H.heldout_dialogue_ids(df_1m)
train_mask, _ = H.split_train_heldout(df, heldout_ids)
df_train = df[train_mask].reset_index(drop=True)

print(f"corpus {CORPUS}: {len(df)} turnos / {df.dialogue_id.nunique()} diálogos")
print(f"held-out: {len(heldout_ids)} diálogos | train: {len(df_train)} turnos / "
      f"{df_train.dialogue_id.nunique()} diálogos")

## 3. Función de entrenamiento (memmap, por modo)

In [ ]:
def train_variant(df_train, emb_memmap, base_cfg, mode, num_layers, num_heads, out_dir,
                  tag, bert_recipe, retrieval=True):
    set_seed(base_cfg.training.seed)
    device = get_device(base_cfg.training.device)

    rng = np.random.default_rng(123)                       # val chica reproducible (curva / best ckpt)
    dids = pd.unique(df_train["dialogue_id"])
    val_dids = set(rng.choice(dids, size=max(1, int(len(dids) * 0.02)), replace=False))
    is_val = df_train["dialogue_id"].isin(val_dids).to_numpy()
    df_tr, df_va = df_train[~is_val], df_train[is_val]

    d = base_cfg.to_dict()
    d["model"]["arch"] = "v2"                               # v2 y v3 comparten arquitectura BERT-fiel
    cfg = Config.from_dict(d)
    cfg.model.attention_mode = mode
    cfg.model.num_layers = num_layers
    cfg.model.num_heads = num_heads
    D = int(emb_memmap.shape[1])
    cfg.model.input_dim = cfg.model.hidden_dim = cfg.model.output_dim = D
    cfg.model.ff_dim = 4 * D                               # 3072 = 4*hidden (intermediate BERT-base)

    losses = resolve_losses_for_mode(cfg.losses, mode)
    if retrieval:
        losses.embedding_retrieval = EmbeddingRetrievalConfig(
            enabled=True, target=("masked" if mode == "bidirectional" else "next_turn"))

    mk = lambda dd: DialogueDataset(dd, emb_memmap, max_turns=cfg.data.max_turns,
                                    num_speakers=cfg.model.num_speakers, lazy=True)
    tr_loader = DataLoader(mk(df_tr), batch_size=cfg.training.batch_size, shuffle=True,
                           num_workers=0, collate_fn=collate_dialogues)
    va_loader = DataLoader(mk(df_va), batch_size=cfg.training.batch_size, shuffle=False,
                           num_workers=0, collate_fn=collate_dialogues)

    model = build_model(cfg.model).to(device)
    if bert_recipe:                                        # receta de BERT
        no_decay = ("bias", "LayerNorm.weight")
        grouped = [
            {"params": [p for n, p in model.named_parameters()
                        if p.requires_grad and not any(nd in n for nd in no_decay)],
             "weight_decay": cfg.training.weight_decay},
            {"params": [p for n, p in model.named_parameters()
                        if p.requires_grad and any(nd in n for nd in no_decay)],
             "weight_decay": 0.0},
        ]
        opt = torch.optim.AdamW(grouped, lr=cfg.training.learning_rate, eps=1e-6)
    else:                                                  # receta del v1 (AdamW uniforme)
        opt = torch.optim.AdamW(model.parameters(), lr=cfg.training.learning_rate,
                                weight_decay=cfg.training.weight_decay)
    total = max(1, len(tr_loader) * cfg.training.epochs)
    sched = build_linear_warmup_scheduler(opt, int(total * cfg.training.warmup_ratio), total)
    nparams = sum(p.numel() for p in model.parameters())
    print(f"[{tag}/{mode}] {type(model).__name__} L={num_layers} A={num_heads} | "
          f"{nparams/1e6:.1f}M params | recipe={'bert' if bert_recipe else 'v1'}", flush=True)

    def move(b):
        o = dict(b)
        o["embeddings"] = b["embeddings"].to(device)
        o["attention_mask"] = b["attention_mask"].to(device)
        if b.get("speaker_ids") is not None:
            o["speaker_ids"] = b["speaker_ids"].to(device)
        return o

    @torch.no_grad()
    def validate():
        model.eval(); set_seed(999); tot = n = 0
        for b in va_loader:
            b = move(b); out = compute_objectives(model, b, losses)
            bs = b["embeddings"].shape[0]; tot += float(out["total"].detach().cpu()) * bs; n += bs
        model.train(); return tot / max(1, n)

    os.makedirs(out_dir, exist_ok=True)
    logf = open(os.path.join(out_dir, "trainlog.jsonl"), "w"); best = float("inf"); t0 = time.time()
    for ep in range(1, cfg.training.epochs + 1):
        model.train(); run = 0.0; te = time.time()
        for i, b in enumerate(tr_loader):
            b = move(b); opt.zero_grad(set_to_none=True)
            loss = compute_objectives(model, b, losses)["total"]; loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.training.gradient_clip_norm)
            opt.step(); sched.step(); run += float(loss.detach().cpu())
            if i % cfg.training.log_interval == 0:
                print(f"[{tag}/{mode}] ep{ep} {i}/{len(tr_loader)} loss={float(loss.detach().cpu()):.4f}", flush=True)
        rec = {"tag": tag, "arch": "v2", "recipe": ("bert" if bert_recipe else "v1"),
               "n_layers": num_layers, "n_heads": num_heads, "lr": cfg.training.learning_rate,
               "mode": mode, "epoch": ep, "train_loss": round(run / max(1, len(tr_loader)), 5),
               "val_loss": round(validate(), 5), "epoch_sec": round(time.time() - te, 1)}
        print("EPOCH", json.dumps(rec), flush=True); logf.write(json.dumps(rec) + "\n"); logf.flush()
        model.save_pretrained(out_dir, training_args=rec)
        if rec["val_loss"] < best:
            best = rec["val_loss"]; model.save_pretrained(os.path.join(out_dir, "best"), training_args=rec)
    cfg.to_yaml(os.path.join(out_dir, "config.yaml"))
    print(f"[{tag}/{mode}] DONE best_val={best:.5f} min={(time.time()-t0)/60:.1f}", flush=True)
    return model

## 4. Entrenar — **receta de BERT** (12 capas), **una tanda por modo**

`lr=1e-4`, AdamW con **weight decay 0 en bias/LayerNorm** y `eps=1e-6` (como BERT), warmup + decay lineal, **`EPOCHS=15`** (tope antes del overfit: v2 convergía AR~ep6 / Bidi~ep11, y con `lr` más bajo el v3 va un poco más lento → 15 da margen), best-by-val.

> ⏱️ **~2,5 h/época en M2 (12 capas)** → ~37 h por modo. Corré **4a (AR)** y, cuando termine, **4b (Bidi)** por separado. `caffeinate -dimsu` + enchufada para que no se duerma.

> 🎯 **No la cortes a mano:** el LR decae a 0 **dentro de las 15 épocas**, así que el modelo anneala bien hasta el final. Dejá que cada tanda complete su schedule (cortar antes = LR sub-decaído = peor `best/`). Si querés cambiar el tope, editá `EPOCHS` arriba **antes** de arrancar.

### 4a. AR (autoregresivo) — corré esta PRIMERO

In [ ]:
# Corré PRIMERO esta (AR). Carpeta NUEVA por versión -> no pisa registros previos.
base_cfg = Config.from_yaml(CONFIG_PATH)
base_cfg.training.epochs = EPOCHS
base_cfg.training.learning_rate = LR
NAME = "contextual-turn-encoder-base"
out_ar = os.path.join(OUT, f"{NAME}-{VER}-ar-{CORPUS}")
assert not os.path.exists(os.path.join(out_ar, "trainlog.jsonl")) or os.environ.get("OVERWRITE"), \
    f"Ya existe {out_ar} — borralo o exportá OVERWRITE=1 para regenerar. (Protege los registros.)"

m_ar = train_variant(df_train, emb, base_cfg, "autoregressive", NLAYERS_OVERRIDE, NHEADS_OVERRIDE,
                     out_ar, tag=VER, bert_recipe=BERT_RECIPE)
print("AR listo ->", out_ar)

### 4b. Bidi (full-context) — corré esta DESPUÉS del AR

In [ ]:
# Corré esta DESPUÉS, cuando termine el AR (si reiniciaste el kernel, re-corré las celdas 1-3 antes).
base_cfg = Config.from_yaml(CONFIG_PATH)
base_cfg.training.epochs = EPOCHS
base_cfg.training.learning_rate = LR
NAME = "contextual-turn-encoder-base"
out_bidi = os.path.join(OUT, f"{NAME}-{VER}-bidi-{CORPUS}")
assert not os.path.exists(os.path.join(out_bidi, "trainlog.jsonl")) or os.environ.get("OVERWRITE"), \
    f"Ya existe {out_bidi} — borralo o exportá OVERWRITE=1 para regenerar. (Protege los registros.)"

m_bidi = train_variant(df_train, emb, base_cfg, "bidirectional", NLAYERS_OVERRIDE, NHEADS_OVERRIDE,
                       out_bidi, tag=VER, bert_recipe=BERT_RECIPE)
print("Bidi listo ->", out_bidi)

## 5. Después

- **Curvas v1 / v2 / v3:** `python training/contextual-turn-encoder-base/plot_full_results.py`.
- **Act-match:** `python packages/conversational-ann/scripts/eval_prelim.py --corpus 1m --phase encode`
  y luego `--phase metric` (despacha por `config.json`).
- Resumen train/eval: `training/contextual-turn-encoder-base/train_eval_results.md`.
- Pesos: no van a git (`models/` gitignored) → asset de un GitHub Release, como el v1.